# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [2]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
#from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [3]:
load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openrouter_base_url = os.getenv('OPENROUTER_BASE_URL')

# Check the key

if not openrouter_api_key:
    print("No API key was found")
elif not openrouter_api_key.startswith("sk"):
    print("An API key was found, but it doesn't start with sk; please check you're using the right key")
else:
    print("API key found and looks good so far!")

# Check the base url

if not openrouter_base_url:
    print("No Base URL was found")
elif not openrouter_base_url.startswith("https://"):
    print("Base URL was found, but it doesn't start with https")
else:
    print("Base URL was found and looks good so far!")


API key found and looks good so far!
Base URL was found and looks good so far!


In [4]:
# constants

MODEL_GPT = 'openai/gpt-5-mini'
MODEL_LLAMA = 'meta-llama/llama-3.2-3b-instruct'

In [5]:
openAI = OpenAI(base_url=openrouter_base_url, api_key=openrouter_api_key);

In [6]:
# user and system prompt

user_prompt = """
Please explain why this code breaks and the solution code:
def calculate_average(numbers)
    total = 0
    for i in range(0, len(numbers)):
        total = total + numbers[i]
    average = total / len(numbers
    return average
"""


system_prompt = """
You will be provided with some code, and a question about the code.
Your job is to explain the code in a way that is easy to understand and why it works.
"""

In [7]:
# Get Llama 3.2 to answer
response = openAI.chat.completions.create(model = MODEL_LLAMA, messages = [{'role':'system','content':f'{system_prompt}'}, {'role':'user','content':f'{user_prompt}'}])
print(response)
display(Markdown(response.choices[0].message.content))

ChatCompletion(id='gen-1771945094-QMLL2mnaEFpMgF96BfuX', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="**Explanation of the original code**\n\nThe original code is designed to calculate the average of a list of numbers. However, it has a few issues that prevent it from working correctly.\n\nHere's what the code does:\n\n1. It initializes a variable `total` to 0, which will be used to accumulate the sum of the numbers in the list.\n2. It loops through the list of numbers using a `for` loop, iterating from the first element (index 0) to the last element (index `len(numbers) - 1`).\n3. Inside the loop, it adds the current number to the `total` variable.\n4. After the loop finishes, it calculates the average by dividing the `total` variable by the length of the list.\n5. However, there's a missing `)` at the end of the `len(numbers)` expression, which is required for Python to interpret it as a complete expression.\n\nThe code will bre

**Explanation of the original code**

The original code is designed to calculate the average of a list of numbers. However, it has a few issues that prevent it from working correctly.

Here's what the code does:

1. It initializes a variable `total` to 0, which will be used to accumulate the sum of the numbers in the list.
2. It loops through the list of numbers using a `for` loop, iterating from the first element (index 0) to the last element (index `len(numbers) - 1`).
3. Inside the loop, it adds the current number to the `total` variable.
4. After the loop finishes, it calculates the average by dividing the `total` variable by the length of the list.
5. However, there's a missing `)` at the end of the `len(numbers)` expression, which is required for Python to interpret it as a complete expression.

The code will break with a `SyntaxError` because Python expects a `)` at the end of the `len(numbers)` expression.

**Corrected code**

Here's the corrected code with the missing `)` added:
```python
def calculate_average(numbers):
    total = 0
    for i in range(0, len(numbers)):
        total = total + numbers[i]
    average = total / len(numbers)
    return average
```
**Explanation of the changes**

To fix the code, we simply added a closing `)` at the end of the `len(numbers)` expression, which tells Python that it's a complete expression and allows the code to run correctly.

**Additional improvement**

While the corrected code is functional, there's a more efficient way to calculate the average using Python's built-in `sum()` function:
```python
def calculate_average(numbers):
    total = sum(numbers)
    average = total / len(numbers)
    return average
```
This version of the code is shorter and more readable, and it's also more efficient because it avoids the need to manually loop through the list of numbers.

In [ ]:
# Get GPT OSS 120b to answer, with streaming
stream = openAI.chat.completions.create(model = MODEL_GPT, messages = [{'role':'system','content':f'{system_prompt}'}, {'role':'user','content':f'{user_prompt}'}], stream=True)
display_handle = display(Markdown(""), display_id=True)
response = ""
for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    update_display(Markdown(response), display_id=display_handle.display_id)



What's wrong
- The function header is missing the colon after the parameter list. That causes a SyntaxError.
- The line that computes average is missing a closing parenthesis: average = total / len(numbers. That also causes a SyntaxError.
- At runtime, if numbers is empty you'll get a ZeroDivisionError.

Minimal fixed version (works in Python 3):
def calculate_average(numbers):
    total = 0
    for i in range(0, len(numbers)):
        total = total + numbers[i]
    average = total / len(numbers)
    return average

Cleaner, more Pythonic and safer versions

1) Simple, idiomatic (uses built-in sum):
def calculate_average(numbers):
    if not numbers:
        raise ValueError("numbers must not be empty")
    return sum(numbers) / len(numbers)

2) If you prefer returning 0 for an empty list instead of raising:
def calculate_average(numbers):
    if not numbers:
        return 0
    return sum(numbers) / len(numbers)

Notes
- In Python 3, / returns a float, so no extra work is needed. In Python 2 you would want to convert one operand to float (e.g. float(len(numbers))) or use from __future__ import division to get true division.
- The built-in sum is faster and clearer than a manual loop.

In [ ]:
# Get Llama 3.2 to answer